In [ ]:
"""
Fused inference: combines checkpoints from BOTH runs.
  Run 1 (original notebook): EfficientNetV2-S (5 folds) + ConvNeXtV2-Tiny (2 folds)
  Run 2 (V2 notebook):       ConvNeXt-Base (3 folds) + EVA-02-Base (1 fold)
Total: 11 checkpoints across 4 architectures.
"""

import os, gc, random, numpy as np, pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import timm
from sklearn.model_selection import StratifiedKFold

try:
    from torch.amp import autocast
    def amp_ctx(): return autocast('cuda')
except ImportError:
    from torch.cuda.amp import autocast
    def amp_ctx(): return autocast()

# ============ CONFIG ============
IS_KAGGLE = os.path.exists('/kaggle/input')

if IS_KAGGLE:
    CKPT_DIR_V1 = '/kaggle/input/datasets/ahmedfakhfahk/models-efficientnet'      # EfficientNetV2-S + ConvNeXtV2-Tiny
    CKPT_DIR_V2 = '/kaggle/input/notebooks/ahmedfakhfahk/face-occlusion-prediction'  # ConvNeXt-Base + EVA-02
    IMAGE_DIR = '/kaggle/input/datasets/ahmedfakhfahk/face-occlusion/face_occlusion_images/Crop_224_5fp_100K'
    TRAIN_CSV = '/kaggle/input/datasets/ahmedfakhfahk/face-occlusion/occlusion_datasets/train.csv'
    TEST_CSV = '/kaggle/input/datasets/ahmedfakhfahk/face-occlusion/occlusion_datasets/test_students.csv'
else:
    CKPT_DIR_V1 = '.'                # best_model_fold*.pth, best_model_convnext_fold*.pth
    CKPT_DIR_V2 = './kaggle_results'  # best_convnext_base.*, best_eva02_base.*
    IMAGE_DIR = './Crop_224_5fp_100K'
    TRAIN_CSV = './occlusion_datasets/train.csv'
    TEST_CSV = './occlusion_datasets/test_students.csv'

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
IMG_SIZE = 224
NUM_WORKERS = 4
BATCH_SIZE = 64
SEED = 42
N_FOLDS = 5

# ============ TWO MODEL ARCHITECTURES ============

# Run 1 head: Dropout + Linear + ReLU (no LayerNorm, no GELU)
class OcclusionModelV1(nn.Module):
    def __init__(self, name, pretrained=False):
        super().__init__()
        self.backbone = timm.create_model(name, pretrained=pretrained, num_classes=0)
        feat = self.backbone.num_features
        self.head = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(feat, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 1),
        )
    def forward(self, x):
        return torch.sigmoid(self.head(self.backbone(x))).squeeze(-1)

# Run 2 (V2) head: LayerNorm + GELU + deeper
class OcclusionModelV2(nn.Module):
    def __init__(self, name, pretrained=False):
        super().__init__()
        self.backbone = timm.create_model(name, pretrained=pretrained, num_classes=0)
        feat = self.backbone.num_features
        self.head = nn.Sequential(
            nn.LayerNorm(feat),
            nn.Dropout(0.3),
            nn.Linear(feat, 512),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(512, 128),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(128, 1),
        )
    def forward(self, x):
        return torch.sigmoid(self.head(self.backbone(x))).squeeze(-1)

# ============ CHECKPOINT REGISTRY ============
CHECKPOINTS = [
    # --- Run 1: EfficientNetV2-S (5 folds, V1 head, ImageNet normalization) ---
    {
        'name': 'tf_efficientnetv2_s.in21k_ft_in1k',
        'model_class': OcclusionModelV1,
        'mean': (0.485, 0.456, 0.406),
        'std': (0.229, 0.224, 0.225),
        'folds': {
            0: os.path.join(CKPT_DIR_V1, 'best_model_fold0.pth'),
            1: os.path.join(CKPT_DIR_V1, 'best_model_fold1.pth'),
            2: os.path.join(CKPT_DIR_V1, 'best_model_fold2.pth'),
            3: os.path.join(CKPT_DIR_V1, 'best_model_fold3.pth'),
            4: os.path.join(CKPT_DIR_V1, 'best_model_fold4.pth'),
        },
    },
    # --- Run 1: ConvNeXtV2-Tiny (2 folds, V1 head, ImageNet normalization) ---
    {
        'name': 'convnextv2_tiny.fcmae_ft_in22k_in1k',
        'model_class': OcclusionModelV1,
        'mean': (0.485, 0.456, 0.406),
        'std': (0.229, 0.224, 0.225),
        'folds': {
            0: os.path.join(CKPT_DIR_V1, 'best_model_convnext_fold0.pth'),
            1: os.path.join(CKPT_DIR_V1, 'best_model_convnext_fold1.pth'),
        },
    },
    # --- Run 2: ConvNeXt-Base (3 folds, V2 head, native normalization) ---
    {
        'name': 'convnext_base.clip_laion2b_augreg_ft_in12k_in1k',
        'model_class': OcclusionModelV2,
        'mean': None,  # will resolve from timm
        'std': None,
        'folds': {
            0: os.path.join(CKPT_DIR_V2, 'best_convnext_base.clip_laion2b_augreg_ft_in12k_in1k_f0.pth'),
            1: os.path.join(CKPT_DIR_V2, 'best_convnext_base.clip_laion2b_augreg_ft_in12k_in1k_f1.pth'),
            2: os.path.join(CKPT_DIR_V2, 'best_convnext_base.clip_laion2b_augreg_ft_in12k_in1k_f2.pth'),
        },
    },
    # --- Run 2: EVA-02-Base (1 fold, V2 head, native normalization) ---
    {
        'name': 'eva02_base_patch14_224.mim_in22k',
        'model_class': OcclusionModelV2,
        'mean': None,
        'std': None,
        'folds': {
            0: os.path.join(CKPT_DIR_V2, 'best_eva02_base_patch14_224.mim_in22k_f0.pth'),
        },
    },
]

# ============ DATASET ============
class FaceDS(Dataset):
    def __init__(self, df, image_dir, transform):
        self.df = df.reset_index(drop=True)
        self.dir = image_dir
        self.tf = transform
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        img = Image.open(f"{self.dir}/{r['filename']}").convert('RGB')
        label = r.get('FaceOcclusion', -1)
        gender = r.get('gender', -1)
        return self.tf(img), torch.tensor(label, dtype=torch.float32), torch.tensor(gender, dtype=torch.float32)

# ============ TTA TRANSFORMS ============
def build_tta(mean, std):
    norm = T.Normalize(mean=mean, std=std)
    val = T.Compose([T.Resize((IMG_SIZE, IMG_SIZE)), T.ToTensor(), norm])
    return [
        val,
        T.Compose([T.Resize((IMG_SIZE, IMG_SIZE)), T.RandomHorizontalFlip(1.0), T.ToTensor(), norm]),
        T.Compose([T.Resize((IMG_SIZE, IMG_SIZE)), T.ColorJitter(brightness=(1.1, 1.1)), T.ToTensor(), norm]),
        T.Compose([T.Resize((IMG_SIZE, IMG_SIZE)), T.ColorJitter(brightness=(0.9, 0.9)), T.ToTensor(), norm]),
        T.Compose([T.Resize((IMG_SIZE, IMG_SIZE)), T.ColorJitter(contrast=(1.1, 1.1)), T.ToTensor(), norm]),
        T.Compose([T.Resize((IMG_SIZE, IMG_SIZE)), T.RandomHorizontalFlip(1.0),
                   T.ColorJitter(brightness=(1.05, 1.05)), T.ToTensor(), norm]),
    ]

# ============ METRICS ============
def weighted_err(pred, target):
    w = 1.0 / 30.0 + target
    return float(np.sum(w * (pred - target) ** 2) / np.sum(w))

def official_score(pred, target, gender):
    mf, mm = gender == 0.0, gender == 1.0
    ef = weighted_err(pred[mf], target[mf]) if mf.sum() else 0.0
    em = weighted_err(pred[mm], target[mm]) if mm.sum() else 0.0
    return (ef + em) / 2.0 + abs(ef - em), ef, em

# ============ INFERENCE ============
@torch.no_grad()
def predict_tta(model_cls, model_name, ckpt_path, df, image_dir, mean, std):
    model = model_cls(model_name, pretrained=False).to(DEVICE)
    model = model.to(memory_format=torch.channels_last)
    sd = torch.load(ckpt_path, map_location=DEVICE, weights_only=True)
    model.load_state_dict(sd, strict=True)
    model.eval()

    ttas = build_tta(mean, std)
    acc = None
    for t_idx, tf in enumerate(ttas):
        ds = FaceDS(df, image_dir, tf)
        loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False,
                            num_workers=NUM_WORKERS, pin_memory=True)
        chunks = []
        for x, _, _ in tqdm(loader, desc=f'TTA {t_idx+1}/{len(ttas)}', leave=False):
            x = x.to(DEVICE, memory_format=torch.channels_last)
            with amp_ctx():
                chunks.append(model(x).float().cpu().numpy())
        p = np.concatenate(chunks)
        acc = p if acc is None else acc + p

    del model; gc.collect(); torch.cuda.empty_cache()
    return acc / len(ttas)

# ============ FOLD SPLITS (must match training — same for both runs) ============
def make_folds(df):
    df = df.copy()
    df['occ_bin'] = pd.cut(df['FaceOcclusion'], bins=10, labels=False)
    df['strat'] = df['gender'].astype(int).astype(str) + '_' + df['occ_bin'].astype(str)
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    df['fold'] = -1
    for f, (_, vidx) in enumerate(skf.split(df, df['strat'])):
        df.loc[vidx, 'fold'] = f
    return df

# ============ MAIN ============
print(f'Device: {DEVICE}')
df_train = pd.read_csv(TRAIN_CSV).dropna().reset_index(drop=True)
df_test = pd.read_csv(TEST_CSV).dropna().reset_index(drop=True)
df_train = make_folds(df_train)
print(f'Train: {len(df_train)} | Test: {len(df_test)}')

# Resolve native normalization for V2 models
for cfg in CHECKPOINTS:
    if cfg['mean'] is None:
        probe = cfg['model_class'](cfg['name'], pretrained=False)
        dc = timm.data.resolve_model_data_config(probe.backbone)
        cfg['mean'], cfg['std'] = dc['mean'], dc['std']
        del probe; gc.collect()

# ============ OOF EVALUATION ============
print('\n' + '='*70)
print('OOF EVALUATION')
print('='*70)

oof_per_model = {}  # key -> (oof_array, mask)

for cfg in CHECKPOINTS:
    name = cfg['name']
    model_cls = cfg['model_class']
    mean, std = cfg['mean'], cfg['std']

    oof = np.zeros(len(df_train), dtype=np.float64)
    mask = np.zeros(len(df_train), dtype=bool)
    found_any = False

    print(f'\n{name} ({model_cls.__name__})')
    for fold_idx, ckpt_path in cfg['folds'].items():
        if not os.path.exists(ckpt_path):
            print(f'  Fold {fold_idx}: SKIP (not found: {os.path.basename(ckpt_path)})')
            continue
        found_any = True
        val_df = df_train[df_train.fold == fold_idx].copy()
        val_indices = val_df.index.values
        print(f'  Fold {fold_idx}: {len(val_df)} val samples...')

        preds = predict_tta(model_cls, name, ckpt_path, val_df, IMAGE_DIR, mean, std)
        oof[val_indices] = preds
        mask[val_indices] = True

        s, ef, em = official_score(preds, val_df['FaceOcclusion'].values, val_df['gender'].values)
        print(f'    score={s:.6f} | err_F={ef:.6f} err_M={em:.6f} | gap={abs(ef-em):.6f}')

    if found_any:
        key = f'{name}_{model_cls.__name__}'
        oof_per_model[key] = (oof, mask)
        m = mask
        if m.sum() > 0:
            s, ef, em = official_score(oof[m], df_train['FaceOcclusion'].values[m], df_train['gender'].values[m])
            print(f'  Overall OOF ({m.sum()} samples): score={s:.6f} | err_F={ef:.6f} err_M={em:.6f}')

# Blended OOF (samples covered by ALL models)
common = np.ones(len(df_train), dtype=bool)
for oof, mask in oof_per_model.values():
    common &= mask

if common.sum() > 0:
    blend = np.mean([oof[common] for oof, mask in oof_per_model.values()], axis=0)
    y = df_train['FaceOcclusion'].values[common]
    g = df_train['gender'].values[common]
    s, ef, em = official_score(blend, y, g)
    print(f'\nBlended OOF ({common.sum()} samples, {len(oof_per_model)} models):')
    print(f'  score={s:.6f} | err_F={ef:.6f} err_M={em:.6f} | gap={abs(ef-em):.6f}')

# ============ TEST PREDICTIONS ============
print('\n' + '='*70)
print('TEST PREDICTIONS')
print('='*70)

all_test_preds = []
for cfg in CHECKPOINTS:
    name = cfg['name']
    model_cls = cfg['model_class']
    mean, std = cfg['mean'], cfg['std']

    fold_preds = []
    for fold_idx, ckpt_path in cfg['folds'].items():
        if not os.path.exists(ckpt_path):
            continue
        print(f'  {name} fold {fold_idx}...')
        pred = predict_tta(model_cls, name, ckpt_path, df_test, IMAGE_DIR, mean, std)
        fold_preds.append(pred)
        print(f'    mean={pred.mean():.4f} std={pred.std():.4f}')

    if fold_preds:
        model_pred = np.mean(fold_preds, axis=0)
        all_test_preds.append(model_pred)
        print(f'  {name} avg: mean={model_pred.mean():.4f}')

# Equal-weight blend
final = np.mean(all_test_preds, axis=0)
final = np.clip(final, 0.0, 1.0)
print(f'\nFinal ensemble ({len(all_test_preds)} models): range [{final.min():.4f}, {final.max():.4f}] mean={final.mean():.4f}')

# Save
sub = pd.DataFrame({'filename': df_test['filename'], 'FaceOcclusion': final, 'gender': 'x'})
sub.to_csv('test_predictions.csv', index=False)
print(f'\nSaved test_predictions.csv ({len(sub)} rows)')
print(sub['FaceOcclusion'].describe())